**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Uncertainty in ML

A model that says '87%' should be right 87% of the time — most deep models aren't. Two sessions on measuring calibration and fixing it with the workhorse tools: temperature scaling and deep ensembles, with a hard look at what happens *off*-distribution.

## 1. Pre-requisites

- [Training Dynamics](./Training_Dynamics.ipynb) (we reuse its spiral testbed).
- [Kernel Methods](./Kernel_Methods.ipynb) S2 — GPs as the calibration gold standard.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3 — the Bayesian frame.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# noisy spirals again — genuine class overlap means genuine aleatoric uncertainty
def spirals(n=3000, noise=0.9, seed=0):
    r = np.random.default_rng(seed)
    t = np.linspace(0.5, 3*np.pi, n//2)
    X, y = [], []
    for cls, ph in [(0, 0.0), (1, np.pi)]:
        X.append(np.stack([t*np.cos(t+ph), t*np.sin(t+ph)], 1) + noise*r.standard_normal((n//2, 2)))
        y.append(np.full(n//2, cls))
    X = np.concatenate(X).astype(np.float32); y = np.concatenate(y).astype(np.int64)
    X = (X - X.mean(0)) / X.std(0)
    idx = r.permutation(n)
    return torch.from_numpy(X[idx]), torch.from_numpy(y[idx])

Xa, ya = spirals()
Xtr, ytr, Xte, yte = Xa[:2000], ya[:2000], Xa[2000:], ya[2000:]

def make_net(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))

def fit(net, epochs=600):     # deliberately overtrained — watch the confidence outrun the accuracy
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for ep in range(epochs):
        for i in range(0, 2000, 200):
            opt.zero_grad()
            Fn.cross_entropy(net(Xtr[i:i+200]), ytr[i:i+200]).backward()
            opt.step()
    return net

---
### 🕐 Session 1 of 2 — *Calibration & Temperature Scaling* (~40 min)
**Goal:** measure whether confidences mean anything; fix miscalibration with one parameter.
**Builds on:** [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 2 (ensembles & OOD).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Calibration & Temperature Scaling</b></summary>

**Timing (~40 min).** 10 min accuracy versus calibration · 10 min reliability diagrams and ECE · 12 min temperature scaling · 8 min reading the result.

**Open by separating two questions students routinely conflate.** *Accuracy* asks how often the model is right. *Calibration* asks whether the number it prints means anything — **when it says 87%, is it right 87% of the time?** A model can be accurate and badly calibrated (right often, but always claiming 99%), or calibrated and inaccurate (right 60% of the time and honestly saying so). Ask which you would rather deploy in a medical triage system; the answer is usually the second, and that reframing motivates the whole session.

**Explain why deep networks are systematically overconfident, since "they just are" is unsatisfying.** Cross-entropy keeps rewarding sharper probabilities long after accuracy has saturated: pushing a correct prediction from 0.9 to 0.99 lowers the loss, so training continues to inflate confidence with no accuracy left to gain. The `fit` function here runs **600 epochs deliberately** for exactly this reason. Modern architectures, batch norm, and long schedules all make it worse — the well-known Guo et al. result is that accuracy improved over the decade while calibration got worse.

**Make the reliability diagram concrete before showing one.** Bin the predictions by stated confidence, and for each bin plot *actual* accuracy against *mean* stated confidence. Perfect calibration is the diagonal. **Below the diagonal is overconfidence** (you claimed 90%, you were right 80%); above is underconfidence. ECE is just the bin-count-weighted average vertical distance — one number summarising the whole picture.

**Flag the two things ECE hides, because it is quoted far too casually.** It depends on the **binning** (10 bins versus 15 gives different numbers), and it is an *average*, so a model can have a small ECE while being badly wrong in the high-confidence bin that actually matters for decisions. Always look at the diagram, not only the scalar.

**Then temperature scaling, and lead with the property that makes it remarkable.** Divide every logit by one scalar $T$. Because softmax is monotone and $T > 0$, the **argmax never changes** — so accuracy is *exactly* preserved, to the last example. You cannot lose anything by applying it. It has one parameter, needs no retraining, and is fitted post hoc on a held-out split by minimising NLL. **A free fix with a guarantee attached is rare; this is one.**

**Be precise about the validation split, since it is the one place this can be done wrong.** $T$ is fitted on `logits[:500]` and *evaluated* on `logits[500:]`. Fitting and reporting on the same data would make any improvement meaningless. Point at those two slices explicitly — students copying this code into their own work get this wrong constantly.

**Set expectations honestly: the improvement here will be modest.** The starting ECE is 0.041, which is mild — the spiral has `noise=0.9` and genuine class overlap, so much of the model's uncertainty is **aleatoric** (irreducible label noise) rather than overconfidence. Temperature scaling can only fix the overconfident part. On a real image classifier trained to convergence, ECE of 0.10–0.15 dropping to under 0.02 is typical, and the visual difference is dramatic. **Say what the demo is and is not showing** rather than overselling a small number.
</details>

## 2. Does 87% Mean 87%?

💡 **Intuition.** Accuracy asks 'how often right?'; **calibration** asks 'when you say 87%, are you right 87% of the time?' The reliability diagram answers it: bin predictions by confidence, plot accuracy per bin against the diagonal. Deep nets trained to convergence sit *below* the diagonal — overconfident — because cross-entropy keeps rewarding sharper probabilities long after accuracy saturates. The embarrassingly effective fix: **temperature scaling** — divide the logits by one scalar $T$ fitted on validation data. It can't change any decision (argmax is $T$-invariant); it only makes the *confidence honest*.

In [2]:
net = fit(make_net(0))
with torch.no_grad():
    logits = net(Xte)
    conf, pred = Fn.softmax(logits, 1).max(1)
acc_overall = (pred == yte).float().mean()

def reliability(conf, pred, yv, bins=10):
    edges = np.linspace(0.5, 1.0, bins+1)
    accs, confs, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if m.sum() > 4:
            accs.append((pred[m] == yv[m]).float().mean().item())
            confs.append(conf[m].mean().item()); ns.append(int(m.sum()))
    return np.array(confs), np.array(accs), np.array(ns)

def ece(conf, pred, yv):
    c, a, n_ = reliability(conf, pred, yv)
    return float(np.sum(n_ * np.abs(c - a)) / n_.sum())

print(f"accuracy {acc_overall:.1%}   ECE (expected calibration error) {ece(conf, pred, yte):.3f}")

accuracy 89.8%   ECE (expected calibration error) 0.041


**What just happened.** Two numbers that answer two different questions: **accuracy 89.8%** — how often the model is right — and **ECE 0.041** — whether the confidence it prints means anything. On average the stated confidence is off by about **4 percentage points**, and for a deep network trained to convergence the error is in the expected direction: overconfident.

**Read ECE as what it literally computes.** Bin the test predictions by stated confidence; in each bin compare mean confidence against actual accuracy; average the gaps weighted by bin population. So 0.041 means "when this model says 90%, it is right about 86% of the time" — small enough to sound harmless, large enough to matter if you are thresholding on confidence to decide whether a human should review the case.

**The overconfidence is manufactured by the training loop, not by bad luck.** Cross-entropy keeps rewarding sharper probabilities after accuracy has saturated: moving a correct prediction from 0.90 to 0.99 lowers the loss even though it changes no decision. `fit` runs **600 epochs deliberately** so the confidence has time to outrun the accuracy. This is the well-documented modern failure mode — accuracy improved across a decade of architecture research while calibration got *worse*.

**But be careful about how much of the 0.041 is fixable.** This spiral is generated with `noise=0.9`, so the classes genuinely overlap and a large share of the model's uncertainty is **aleatoric** — irreducible label noise that no amount of calibration removes. A perfectly calibrated model on this data would still be wrong about 10% of the time and would *correctly* say so. Temperature scaling can only address the part that comes from overconfidence, which is why the improvement in the next cell is modest rather than dramatic.

**Note two limitations of ECE before quoting it anywhere.** It is **binning-dependent** — 10 bins and 15 bins give different numbers for identical predictions — and it is an **average**, so a model can post a small ECE while being badly miscalibrated in exactly the high-confidence bin that drives automated decisions. The `reliability` helper here also drops bins with fewer than 5 points, which quietly discards sparse regions. **Look at the diagram, not only the scalar.**

**Finally, keep the two numbers separate in your head, because they trade off independently.** A model can be accurate and dishonest (right often, always claiming 99%) or honest and weak (right 60% of the time and saying 60%). For a system that hands borderline cases to a human, the second is deployable and the first is dangerous. **Accuracy tells you how good the model is; calibration tells you whether you can act on what it says.**

In [3]:
# fit T on a validation split by minimizing NLL — one parameter, no retraining
T = torch.ones(1, requires_grad=True)
opt_t = torch.optim.LBFGS([T], lr=0.1, max_iter=50)
val_logits, val_y = logits[:500].detach(), yte[:500]
def closure():
    opt_t.zero_grad()
    loss = Fn.cross_entropy(val_logits / T.clamp(min=0.05), val_y)
    loss.backward(); return loss
opt_t.step(closure)

with torch.no_grad():
    conf_T, pred_T = Fn.softmax(logits[500:] / T, 1).max(1)
conf_raw, pred_raw = Fn.softmax(logits[500:], 1).max(1)

fig, ax = plt.subplots(figsize=(4.6, 4))
for name, (c_, p_) in [("raw", (conf_raw, pred_raw)), (f"T = {T.item():.2f}", (conf_T, pred_T))]:
    cc, aa, _ = reliability(c_, p_, yte[500:])
    ax.plot(cc, aa, "o-", label=f"{name} (ECE {ece(c_, p_, yte[500:]):.3f})")
ax.plot([0.5, 1], [0.5, 1], "k--", linewidth=0.8, label="perfect calibration")
ax.set_xlabel("stated confidence"); ax.set_ylabel("actual accuracy"); ax.legend(fontsize=8)
ax.set_title("reliability diagram: temperature drags the curve onto the diagonal")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2700996/40931829.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two reliability curves against the dashed diagonal. The raw curve sits **below** it — the model claims more confidence than it earns — and the temperature-scaled curve is dragged toward it, with a lower ECE in the legend. One scalar, fitted by LBFGS on 500 held-out points, no retraining.

**The property that makes temperature scaling remarkable is that it cannot cost you anything.** Softmax is monotone and $T > 0$, so dividing every logit by $T$ **never changes the argmax**. Accuracy is preserved *exactly* — not approximately, not on average, but example by example. All that changes is how sharp the probabilities are. A post-hoc fix with one parameter and a guarantee attached is rare; take it whenever you have a validation split.

**Note where $T$ was fitted and where it was evaluated, because this is the one way to get it wrong.** $T$ comes from `logits[:500]`; both curves are computed on `logits[500:]`. Fit and report on the same data and any improvement is circular. Students copying this pattern get it wrong constantly, and the fix is the two slices visible in the code.

**Read $T$ itself as a diagnostic.** $T > 1$ softens the logits and means the model was **overconfident**; $T < 1$ sharpens them and means it was underconfident. Trained-to-convergence networks essentially always land above 1 — cross-entropy keeps rewarding sharper probabilities long after accuracy has stopped improving, so 600 epochs buy confidence rather than correctness.

**Be honest that the improvement here is modest, and say why.** The starting ECE is 0.041, which is mild to begin with. The spiral is generated with `noise=0.9`, so the classes genuinely overlap and much of the model's uncertainty is **aleatoric** — irreducible label noise that is *correct* to report and that no calibration method should remove. Temperature scaling only addresses the overconfident component. On a ResNet trained on CIFAR you would typically see ECE fall from 0.10–0.15 to under 0.02 and the raw curve bow visibly away from the diagonal; this demo shows the mechanism on a problem that was not badly broken.

**And be clear about the fundamental limitation, which Session 2 exists to probe.** A single global $T$ applies the same correction everywhere in input space. It cannot say "I am well calibrated near the training data and hopeless 5 units away" — that would require the correction to depend on $x$. **Temperature scaling fixes the *average* honesty of a model on the distribution it was calibrated on, and does nothing off it.** Every confidence number in this notebook, raw or scaled, is conditional on the test point coming from the same world as the training data.

---
### 🕐 Session 2 of 2 — *Ensembles & the Out-of-Distribution Problem* (~40 min)
**Goal:** average independently-trained nets for better uncertainty; test where all bets are off.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Ensembles & the Out-of-Distribution Problem</b></summary>

**Timing (~40 min).** 10 min the ensemble idea · 10 min the in-distribution result (a null one) · 15 min the OOD scan · 5 min the honest conclusion.

**Introduce deep ensembles as belief-by-samples rather than as a trick.** Train the same architecture from $k$ different seeds and average the probabilities. Where the data constrains the function, members agree; where it does not, they disagree — and **that disagreement is an uncertainty signal a single model structurally cannot produce**. It is a crude Bayesian posterior ([Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3) represented by samples, exactly like the [particle filter](../Intro_Time_Series/Beyond_Kalman.ipynb). Cost: $k\times$ everything, and it remains the strongest practical baseline in the field.

**Prepare the room for a null result in-distribution, and do not fake it.** The ensemble comes out at **89.4% / ECE 0.041** against the single model's **89.8% / 0.041** — no better on either metric. Say so plainly. The reason is diagnosable: the bootstrap resampling gives each member only ~63% of the unique training points, and on a task where the model is already near the aleatoric floor, that lost data costs about as much as the averaging gains. **A demo that reports its own null result teaches more than one that quietly reruns until it wins.**

**Then pivot to where ensembles actually earn their keep, and make the pivot explicit.** In-distribution calibration is temperature scaling's job — one parameter, no retraining, guaranteed not to hurt. Ensembles cost 5× and are for the problem temperature *cannot* touch: **behaviour off the training distribution**. That framing turns the null result into the setup for Session 2's real content.

**Set up the OOD scan with the structural argument first.** Softmax **must** sum to one. There is no "none of the above" output, so a network handed a point from another world still allocates all its probability among the classes it knows, and can do so with total confidence. This is not a bug in training; it is what the output layer is. Ask the room what a model *should* say about a point 5 units from any data, then show what it does say.

**Read the two panels together.** Single-model max-softmax is confident nearly everywhere, including the far corners. Ensemble entropy rises in regions the members disagree about. The improvement is real and visible — and then the numbers spoil the story, which is the point.

**Give the numbers their full weight: only 9% of the off-map area is flagged, and 90% has both the ensemble *and* the single net confidently committed.** Ensembles **mitigate** OOD overconfidence; they do not solve it. The mechanism explains why: all five members share an architecture, a training set, and an inductive bias, so far from the data they tend to extrapolate the *same* way. Independent seeds buy independence of initialisation, not independence of assumption.

**Balance that with the flagged points, which are a genuine success.** At $(+3.1, +1.6)$ the single net reports **100.0%** confidence while the ensemble entropy is **0.69 = $\log 2$**, the theoretical maximum for two classes — perfect disagreement, a complete "I don't know". A single model cannot express that value at all. So the tool works where it works; the failure is coverage, not capability.

**Close on the practical ordering.** Calibrate first — temperature is nearly free and cannot hurt. Ensemble when the stakes justify 5×. And treat every confidence, raw or calibrated or ensembled, as **conditional on being on-distribution**; detecting "off the map" is a separate problem with its own literature, and disagreement is a first tool rather than a solution. The [GP](./Kernel_Methods.ipynb) remains the standard these methods chase, and its own calibration audit came in at 92% — nobody in this area is doing better than "usefully imperfect".
</details>

## 3. Deep Ensembles

💡 **Intuition.** Train the same architecture from $k$ different random seeds and *average the probabilities*. Where the data constrains the function, the members agree; where it doesn't, they disagree — and that **disagreement is an uncertainty signal** that single-model confidence simply doesn't carry. It's a crude Bayesian posterior ([Estimation Theory S3](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb)), cousin to the [particle filter](../Intro_Time_Series/Beyond_Kalman.ipynb): represent belief with samples. Cost: $k\times$ everything — and it remains the strongest practical baseline in the field.

In [4]:
def fit_boot(seed):
    torch.manual_seed(seed)
    idx = torch.randint(0, 2000, (2000,))            # bootstrap: each member sees a different resample
    net_b = make_net(seed)
    opt = torch.optim.Adam(net_b.parameters(), lr=2e-3)
    for ep in range(600):
        for i in range(0, 2000, 200):
            j = idx[i:i+200]
            opt.zero_grad(); Fn.cross_entropy(net_b(Xtr[j]), ytr[j]).backward(); opt.step()
    return net_b

members = [fit_boot(s) for s in range(5)]
with torch.no_grad():
    probs = torch.stack([Fn.softmax(m(Xte), 1) for m in members])
p_ens = probs.mean(0)
conf_e, pred_e = p_ens.max(1)
print(f"single model: acc {(pred == yte).float().mean():.1%}   ECE {ece(conf, pred, yte):.3f}")
print(f"5-ensemble:   acc {(pred_e == yte).float().mean():.1%}   ECE {ece(conf_e, pred_e, yte):.3f}")

single model: acc 89.8%   ECE 0.041
5-ensemble:   acc 89.4%   ECE 0.041


**What just happened.** Five independently trained networks, averaged — and **nothing improved**:

| | accuracy | ECE |
|---|---|---|
| single model | 89.8% | 0.041 |
| 5-ensemble | **89.4%** | **0.041** |

Five times the training cost, five times the inference cost, and the accuracy is slightly *lower* while the calibration error is identical. **This is a null result, and it is worth reporting as one.**

**The cause is diagnosable, and it is in the code.** `fit_boot` draws a **bootstrap resample** — `torch.randint(0, 2000, (2000,))` — so each member sees only about $1 - e^{-1} \approx 63\%$ of the unique training points. Bootstrapping buys diversity between members and pays for it in data per member. On a task already sitting near its aleatoric floor, the data lost costs about as much as the averaging gains, and the two effects cancel. Drop the bootstrap and train five members on the full set with different seeds only, and the accuracy typically edges up while diversity drops.

**The 0.4-point accuracy difference is also within noise.** On 1000 test points the binomial standard error is about 1%, so 89.8% versus 89.4% is well under half a standard error — four examples. **Do not read a ranking into it in either direction.** The honest statement is that on this task, at this scale, the ensemble bought nothing measurable.

**So say what ensembles are actually for, because it is not this.** In-distribution calibration is temperature scaling's job: one parameter, no retraining, provably no accuracy cost. Paying 5× to match it is a bad trade. **Ensembles earn their keep on the problem temperature scaling structurally cannot touch** — behaviour *off* the training distribution, where a single model has no mechanism for doubt at all. The next cell is where that shows up.

**And keep the mechanism in view, since it is what makes the off-distribution case different.** A single softmax outputs one number and has no way to express "the data does not determine this". Five models that agree where the data constrains them and diverge where it does not carry an extra signal — **disagreement** — that no single model can produce, at any confidence value. This cell shows that the signal is worthless where the data is dense; the next shows where it becomes the only thing you have.

**One framing worth keeping.** This is a crude Bayesian posterior represented by samples — the same idea as the [particle filter](../Intro_Time_Series/Beyond_Kalman.ipynb) in the time-series track, where belief is carried by a population rather than a formula. Five samples is a very coarse posterior, which is part of why the next result is only partly good news.

## 4. Off the Map

💡 **Intuition.** The dirty secret of every confidence score: it is only meaningful **on the distribution the model was trained on**. Show the network a point from a different world and softmax still prints a confident number — softmax *must* sum to one; it has no 'none of the above'. Ensemble disagreement (predictive entropy) at least *rises* off-distribution. Compare both on points far outside the spiral:

In [5]:
# scan the plane: single-model confidence vs ensemble entropy
g = np.linspace(-4, 4, 200)
GX, GY = np.meshgrid(g, g)
grid = torch.tensor(np.stack([GX.ravel(), GY.ravel()], 1), dtype=torch.float32)
with torch.no_grad():
    p_single = Fn.softmax(net(grid), 1).max(1).values.reshape(GX.shape)
    pg = torch.stack([Fn.softmax(m(grid), 1) for m in members]).mean(0)
    ent = (-pg * pg.clamp(min=1e-9).log()).sum(1).reshape(GX.shape)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
im0 = axes[0].contourf(GX, GY, p_single, levels=20, cmap="RdYlGn")
axes[0].set_title("single net max-softmax:\nCONFIDENT even far from any data")
im1 = axes[1].contourf(GX, GY, ent, levels=20, cmap="RdYlGn_r")
axes[1].set_title("ensemble predictive entropy:\nuncertainty rises off the spiral")
for ax in axes:
    ax.scatter(*Xtr[:500].T, s=1, c="k", alpha=0.3)
plt.colorbar(im0, ax=axes[0]); plt.colorbar(im1, ax=axes[1])
plt.tight_layout(); plt.show()

# find the off-map points the ensemble actually flags (and admit the ones it doesn't)
radius = np.hypot(GX, GY)
off_map = radius > 3.0
ent_np, ps_np = ent.numpy(), p_single.numpy()
flagged = (ent_np > 0.35) & off_map
fooled  = (ent_np < 0.05) & off_map & (ps_np > 0.99)
print(f"off-map area flagged by ensemble entropy (>0.35): {flagged.sum()/off_map.sum():.0%}")
print(f"off-map area where ensemble AND single net are both confidently wrong-headed: {fooled.sum()/off_map.sum():.0%}")
iy, ix = np.unravel_index(np.argsort(-ent_np*off_map, axis=None)[:3], ent_np.shape)
far = torch.tensor(np.stack([GX[iy, ix], GY[iy, ix]], 1), dtype=torch.float32)
with torch.no_grad():
    sm = Fn.softmax(net(far), 1).max(1).values
    pe = torch.stack([Fn.softmax(m(far), 1) for m in members]).mean(0)
    he = (-pe * pe.clamp(min=1e-9).log()).sum(1)
for i in range(3):
    print(f"flagged point ({far[i,0]:+.1f}, {far[i,1]:+.1f}): single-net confidence {sm[i]:.1%}   ensemble entropy {he[i]:.2f} (max {np.log(2):.2f})")

off-map area flagged by ensemble entropy (>0.35): 9%
off-map area where ensemble AND single net are both confidently wrong-headed: 90%
flagged point (+3.1, +1.6): single-net confidence 100.0%   ensemble entropy 0.69 (max 0.69)
flagged point (+3.9, +2.3): single-net confidence 100.0%   ensemble entropy 0.69 (max 0.69)
flagged point (+3.7, +2.1): single-net confidence 100.0%   ensemble entropy 0.69 (max 0.69)


/tmp/ipykernel_2700996/1850092308.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The plane, painted twice. Single-model max-softmax is **confident almost everywhere**, including the far corners where no training point has ever been. Ensemble entropy rises in some off-spiral regions — visible improvement — and then the numbers arrive and complicate the story:

- off-map area (radius > 3) flagged by ensemble entropy: **9%**
- off-map area where the ensemble *and* the single net are both confident: **90%**

**Take the good news first, because it is real.** At $(+3.1, +1.6)$ the single network reports **100.0%** confidence while the ensemble entropy is **0.69**, which is exactly $\log 2$ — the theoretical maximum for two classes, a perfect coin flip, a complete "I do not know". A single softmax has no way to produce that statement at any confidence value. Where the members disagree, the ensemble says so unambiguously.

**Now the bad news, which is the honest headline: 9% coverage.** Nine tenths of the off-map region is territory where **all five members agree confidently about a point none of them has any basis to judge**. Ensembles *mitigate* out-of-distribution overconfidence; they do not solve it, and a 9% detection rate would not pass as a safety mechanism anywhere.

**The mechanism explains the failure and is worth stating.** The five members share an architecture, a training set, a loss, and an inductive bias. Different seeds buy independence of *initialisation* — not independence of *assumption*. Far from the data all five extrapolate the same way, because ReLU networks extend their outermost linear pieces to infinity and those pieces are largely determined by the data, not the seed. **Disagreement requires the members to be wrong differently, and nothing forced them to be.**

**Underneath both panels is a structural fact about the output layer.** Softmax **must** sum to one. There is no "none of the above" class, so any input — a spiral point, a point 5 units away, a photograph, random noise — has its probability mass distributed among the two classes the model knows. **Confidence is a statement about which class, never about whether the question makes sense.** That is not a training failure to be fixed with more epochs; it is what the architecture computes.

**Which sets the practical rule this workshop exists to deliver.** Every confidence number — raw, temperature-scaled, or ensembled — is **conditional on the input being on-distribution**, and nothing in the model verifies that condition. Detecting "off the map" is its own problem with its own literature (density estimation, Mahalanobis distance in feature space, deep evidential methods, explicit OOD training), and ensemble disagreement is a first tool rather than an answer.

**One caveat on the numbers themselves.** The 9% and 90% figures depend on the arbitrary thresholds `ent > 0.35` and `ent < 0.05 & p > 0.99`, and on defining "off-map" as radius > 3.0. Move those and the percentages move. The qualitative conclusion — most of the off-map plane is confidently classified by both methods — is robust to any reasonable choice, and it is worth checking that yourself rather than taking the two printed numbers at face value.

**For contrast, recall the [GP](./Kernel_Methods.ipynb) from the kernel workshop.** Its posterior variance returns to the *prior* far from data, automatically, because the formula subtracts only what the data explains. That is uncertainty by construction rather than by disagreement — and even it audited at 92% coverage rather than 95%. Nobody in this area is doing better than usefully imperfect; knowing which imperfection you have is the skill.

## 5. Conclusion

Calibrate before you trust (temperature is nearly free); ensemble when the stakes justify 5×; and treat *all* confidences as conditional on being on-distribution — detecting 'off the map' is its own problem, and disagreement is your first tool — but as the area numbers show, it flags only *part* of the off-map world: ensembles mitigate OOD overconfidence, they do not solve it. The [GP](./Kernel_Methods.ipynb) remains the standard these methods chase.

---
## Where next

- [Kernel Methods](./Kernel_Methods.ipynb) S2 — calibrated uncertainty by construction.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-as-samples, the filtering version.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — what 'well-calibrated' means formally.